# DATA 5310 – Earthquake Risk and Seismic Retrofits in Seattle  

### 1 Data Overview
We work with:

- City of Seattle **building permits** (to identify seismic retrofit projects)  
- **Community Reporting Areas (CRA)** boundaries (to summarize patterns by neighborhood)  
- **Unreinforced Masonry Buildings (URM)** data (high-risk building stock)  
- **Environmentally Critical Areas (ECA)** – especially slide-prone areas  
- **Block-level population estimates** (to normalize retrofit activity by residents)

---

## 2 Guiding Questions for Visualization

1. **Time trends** – How has the number of seismic retrofit permit applications changed over time?  
2. **Neighborhood patterns** – Which CRAs show the highest retrofit intensity (per 10,000 residents and as a share of all permits)?  
3. **URM exposure and vulnerability** – Where are URM buildings located, and how are they distributed by vulnerability class?  
4. **Hazard overlap** – Where do URM clusters overlap with slide-prone ECA areas?  
5. **Risk vs mitigation** – At the CRA level, how does relative risk (URM + hazards) compare to relative mitigation (retrofit intensity and URM retrofit coverage)?

> **Tip:** Run the notebook top to bottom after updating paths if your files are in a different folder.


## 3 Loading and Processing the Data

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import altair as alt

from cra_data_pipeline import compile_cra_stats, clean_permits_data, clean_urm_data

In [2]:
seattle_census_data_path = 'data/OFM_SAEP_BLOCK20_ESTIMATES_SEATTLE_-7113746441103743061.geojson'
cras_path = 'data/CITYPLAN_CRA_-6672415173103925082.geojson'
liquefaction_areas_path = 'data/Environmentally_Critical_Areas_ECA_Liquefaction.geojson'
slide_areas_path = 'data/Environmentally_Critical_Areas_ECA_PotentialSlide.geojson'
urm_path = 'data/Unreinforced_Masonry_Buildings_(URM).geojson'
permits_path = 'data/Building_Permits_20251204.csv'

In [3]:
# call with explicit project data paths (files in `data/` folder or update as needed)
cra = compile_cra_stats(
    seattle_census_data_path=seattle_census_data_path,
    cras_path=cras_path,
    liquefaction_areas_path=liquefaction_areas_path,
    slide_areas_path=slide_areas_path,
    urm_path=urm_path,
    permits_path=permits_path,
    permits_file_fmt='csv',
)
print('CRA stats rows:', len(cra))
cra.head()


Adding columns for slide risk, liquefaction risk, and community reporting areas.
CRA stats rows: 53


,OBJECTID,CRA_NO,CRA_GRP,DETL_NAMES,NEIGHDIST,AREA_ACRES,AREA_SQMI,SE_ANNO_CAD_DATA,DISPLAY_NAME,geometry,...,BLDG_PERMIT_COUNT,AVG_EST_PROJECT_COST,AVG_COMPLETION_TIME_DAYS,RETROFIT_PERMIT_COUNT,population,retrofit_share_permits,retrofit_rate_per_10k,risk_index,mitigation_index,GEN_ALIAS
0,1,1.1,1,"Arbor Heights, Brace Point, Endolyne, Arroyo",Southwest,782.825789,1.223165,,CRA - Arbor Heights,"POLYGON ((1254727.437 193486.154, 1254770.673 ...",...,1731,164772.482513,384.290012,27,6324.316,0.015598,42.692364,0.000000,0.290193,Arbor Heights
1,3,1.2,1,"Fauntleroy, Gatewood, Morgan Junction, Lincoln...",Southwest,1396.262454,2.181660,,CRA - Fauntleroy-Seaview,"POLYGON ((1253011.561 208604.272, 1253348.65 2...",...,4666,195636.749550,381.761607,86,15429.556,0.018431,55.737184,0.092417,0.394929,Fauntleroy/Seaview
2,5,1.3,1,"Seaview, Fairmount Park, Morgan Junction, Gene...",Southwest,1449.328856,2.264576,,CRA - West Seattle Junction-Genesee Hill,"POLYGON ((1254202.473 216794.4, 1254202.88 216...",...,5383,419103.801985,344.932039,160,24717.949,0.029723,64.730290,0.142180,0.467556,West Seattle Junction/Genesee Hill
3,6,1.4,1,"Alki, Admiral",Southwest,896.066891,1.400105,,CRA - Alki-Admiral,"POLYGON ((1257068.2 220992.2, 1257068.4 220991...",...,3792,312066.615224,388.000000,69,12033.701,0.018196,57.338968,0.016588,0.407789,Alki/Admiral
4,8,2.1,2,"North Delridge, Pigeon Point, Avalon, Luna Par...",Delridge Neighborhoods,1612.818519,2.520029,,CRA - North Delridge,"MULTIPOLYGON (((1265887.699 207663.314, 126578...",...,2596,653390.770435,422.244318,25,6922.164,0.009630,36.115874,0.033175,0.237391,North Delridge


In [4]:
permits = clean_permits_data(
    data_path=permits_path,
    data_file_fmt='csv',
    liquefaction_areas_path=liquefaction_areas_path,
    slide_areas_path=slide_areas_path,
    cras_path=cras_path,
)
print('Permit rows:', len(permits))
permits.head()

Adding columns for slide risk, liquefaction risk, and community reporting areas.
Permit rows: 185065


,PermitNum,PermitClass,PermitClassMapped,PermitTypeMapped,PermitTypeDesc,Description,EstProjectCost,AppliedDate,ReadyToIssueDate,IssuedDate,...,liquefaction_prone,slide_prone,is_in_cra,CRA_NO,CRA_NAME,X,Y,year,month,year_month
0,3001672-EX,Multifamily,Residential,ECA and Shoreline Exemption/Street Improvement...,Environmentally Critical Area Exemption,Exception/Exemption Request for: Land Use Appl...,NaN,NaT,NaN,NaT,...,False,False,True,8.3,Cedar Park/Meadowbrook,-122.292145,47.720335,NaN,NaN,NaT
1,3002989-EX,Multifamily,Residential,ECA and Shoreline Exemption/Street Improvement...,Environmentally Critical Area Exemption,Exception/Exemption Request for: Council land ...,NaN,NaT,NaN,NaT,...,False,False,True,8.3,Cedar Park/Meadowbrook,-122.296016,47.730305,NaN,NaN,NaT
2,3003217-EX,Commercial,Non-Residential,ECA and Shoreline Exemption/Street Improvement...,Shoreline Exemption,Exception/Exemption Request for: Land use perm...,NaN,NaT,NaN,NaT,...,True,False,True,13.2,Downtown Commercial Core,-122.334383,47.601971,NaN,NaN,NaT
3,3003295-EX,Multifamily,Residential,ECA and Shoreline Exemption/Street Improvement...,Environmentally Critical Area Exemption,Exception/Exemption Request for: Cancel per cu...,NaN,NaT,NaN,NaT,...,False,False,True,11.1,Fremont,-122.349362,47.650242,NaN,NaN,NaT
4,3003402-EX,NaN,NaN,ECA and Shoreline Exemption/Street Improvement...,Environmentally Critical Area Exemption,Exception/Exemption Request for: PROJECT CANCE...,NaN,NaT,NaN,NaT,...,False,True,True,12.1,Magnolia,-122.414878,47.641339,NaN,NaN,NaT


In [5]:
urm = clean_urm_data(urm_path, cras_path)
print("URM records:", urm.shape[0])
print(urm["VULNERABILITY_CLASSIFICATION"].value_counts(dropna=False))

EPSG:4326
EPSG:4326
URM records: 1111
VULNERABILITY_CLASSIFICATION
Medium      794
High        202
Critical    114
None          1
Name: count, dtype: int64


In [6]:
# polygons for mapping
eca_slide = gpd.read_file(slide_areas_path).to_crs(epsg=4326)
eca_liquefaction = gpd.read_file(liquefaction_areas_path).to_crs(epsg=4326)

In [7]:

# set variables expected by downstream notebook cells
cra_permits = cra[['CRA_NO', 'GEN_ALIAS']].copy()
cra_permits['total_permits'] = cra.get('BLDG_PERMIT_COUNT', 0)
cra_permits['retrofit_permits'] = cra.get('RETROFIT_PERMIT_COUNT', 0)
cra_risk_mit = cra[['CRA_NO','GEN_ALIAS','risk_score','urm_retrofit_share','risk_index','mitigation_index','retrofit_rate_per_10k']].copy()

## 4 Results and Visualizations

### 4.1 Temporal Patterns of Retrofit Activity

We look at retrofit activity over time at **year** and **month** resolution, and compare to all permits.


In [8]:
# -----------------------------------------------------------------------------
# 2.1 Yearly counts and retrofit share
# -----------------------------------------------------------------------------
permits_valid = permits[~permits['AppliedDate'].isna()]
retrofits_valid = permits_valid[permits_valid['topic'] == 'retrofit']

yearly_all = (
    permits_valid.groupby("year")
    .size()
    .reset_index(name="permit_count")
)

yearly_retro = (
    retrofits_valid.groupby("year")
    .size()
    .reset_index(name="retrofit_count")
)

yearly = yearly_all.merge(yearly_retro, on="year", how="left")
yearly["retrofit_count"] = yearly["retrofit_count"].fillna(0)
yearly["retrofit_share"] = yearly["retrofit_count"] / yearly["permit_count"]

yearly

,year,permit_count,retrofit_count,retrofit_share
0,1986.0,1,0.0,0.000000
1,1988.0,1,0.0,0.000000
2,1989.0,2,0.0,0.000000
3,1990.0,6,0.0,0.000000
4,1992.0,8,0.0,0.000000
5,1994.0,2,0.0,0.000000
6,1995.0,4,0.0,0.000000
7,1996.0,5,0.0,0.000000
8,1997.0,5,0.0,0.000000
9,1998.0,7,1.0,0.142857


In [9]:
# Line chart: yearly retrofit count
alt.Chart(yearly).mark_line(point=True).encode(
    x=alt.X("year:O", title="Year"),
    y=alt.Y("retrofit_count:Q", title="Retrofit applications"),
    tooltip=["year", "retrofit_count", "permit_count", alt.Tooltip("retrofit_share:Q", format=".2%")],
).properties(
    width=600,
    height=300,
    title="Yearly Seismic Retrofit Applications",
)

alt.Chart(...)

In [10]:
# Area chart: retrofit share over time
alt.Chart(yearly).mark_area().encode(
    x=alt.X("year:O", title="Year"),
    y=alt.Y("retrofit_share:Q", title="Retrofits as share of all permits", axis=alt.Axis(format="%")),
    tooltip=["year", alt.Tooltip("retrofit_share:Q", format=".2%")],
).properties(
    width=600,
    height=300,
    title="Share of All Permits That Are Seismic Retrofits",
)

alt.Chart(...)

In [11]:
# Monthly dynamics: heatmap by year-month

monthly_retro = (
    retrofits_valid.groupby("year_month")
    .size()
    .reset_index(name="retrofit_count")
)

monthly_retro["year"] = monthly_retro["year_month"].dt.year.astype(str)
monthly_retro["month"] = monthly_retro["year_month"].dt.month

alt.Chart(monthly_retro).mark_rect().encode(
    x=alt.X("month:O", title="Month"),
    y=alt.Y("year:O", title="Year"),
    color=alt.Color("retrofit_count:Q", title="Retrofit count"),
    tooltip=["year_month:T", "retrofit_count:Q"],
).properties(
    width=500,
    height=400,
    title="Heatmap of Retrofit Applications by Year and Month",
)

alt.Chart(...)

## References